In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sqlalchemy import create_engine

engine = create_engine("postgresql://berfinkilinc@localhost/pos_db")

In [ ]:
query= "SELECT * FROM pos_terminal_features"

df = pd.read_sql(query, engine)
print(f'veri çekildi : {len(df)}')

In [ ]:
correlations = df.corr(numeric_only=True)["risk_flag"].sort_values(key=abs, ascending=False)
print(correlations)

In [ ]:
if "risk_flag" in df.columns:

    leakage_cols = [
        "risk_flag", "terminal_id", "sector", "recent_device_fault", "sector_deviation"
    ]
    X = df.drop(columns= leakage_cols, errors= "ignore")
    y= df["risk_flag"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print("tamamdır")
else:
    print("risk_flag bulunamadı, sütun eksik.")

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score


In [ ]:
mlflow.set_experiment("POS_Arıza_Risk_Tahmini")

with mlflow.start_run():
    n_estimators = 100 #r andom foresttaki ağaç sayısı
    max_depth = 5 # leaf derinliği (çok derin olursa sıkıntı)

    # parametleri mflow a logluyoruz
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    # model kurulumu
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # tahminler
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # metrikler

    roc_auc = roc_auc_score(y_test,y_prob)

    #mflow a loglanması

    mlflow.log_metric("roc_auc", roc_auc)

    print(f"model eğitildi. roc-auc scoru: {roc_auc:.4f}")
    print("\nsinfilandirma rapuro:")
    print(classification_report(y_test,y_pred))

    # modeli mlflow a kaydet

    mlflow.sklearn.log_model(model, "random_forest_model")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

mlflow.set_experiment("POS_Arıza_Risk_Tahmini")

with mlflow.start_run(run_name="logistic_regression"):
    # Lojistik regresyon, feature'ların ölçeğine (büyüklüğüne) duyarlıdır
    # (örn. overall_volume binlerce, failure_ratio 0-1 arası) -> standartlaştırıyoruz
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Hiperparametreler
    C = 1.0  # regularization gücü (küçük C = daha güçlü regularization)

    mlflow.log_param("model_type", "logistic_regression")
    mlflow.log_param("C", C)

    model_lr = LogisticRegression(C=C, class_weight="balanced", random_state=42, max_iter=1000)
    model_lr.fit(X_train_scaled, y_train)

    y_pred_lr = model_lr.predict(X_test_scaled)
    y_prob_lr = model_lr.predict_proba(X_test_scaled)[:, 1]

    roc_auc_lr = roc_auc_score(y_test, y_prob_lr)
    mlflow.log_metric("roc_auc", roc_auc_lr)

    print(f"Model eğitildi (Logistic Regression). roc-auc scoru: {roc_auc_lr:.4f}")
    print("\nsinifilandirma raporu:")
    print(classification_report(y_test, y_pred_lr))

    mlflow.sklearn.log_model(model_lr, "logistic_regression_model")

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

mlflow.set_experiment("POS_Arıza_Risk_Tahmini")

with mlflow.start_run(run_name="gradient_boosting"):
    n_estimators = 100
    max_depth = 3          # GBM'de ağaçlar genelde sığ tutulur (3-5), RF'den farklı olarak
    learning_rate = 0.1    # her ağacın öğrenmeye katkısı ne kadar "yavaş/temkinli" olsun

    mlflow.log_param("model_type", "gradient_boosting")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("learning_rate", learning_rate)

    model_gb = GradientBoostingClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        random_state=42
    )
    model_gb.fit(X_train, y_train)  # GBM ağaç bazlı olduğu için scaling gerekmiyor, ham X_train yeterli

    y_pred_gb = model_gb.predict(X_test)
    y_prob_gb = model_gb.predict_proba(X_test)[:, 1]

    roc_auc_gb = roc_auc_score(y_test, y_prob_gb)
    mlflow.log_metric("roc_auc", roc_auc_gb)

    print(f"Model eğitildi (Gradient Boosting). roc-auc scoru: {roc_auc_gb:.4f}")
    print("\nsinifilandirma raporu:")
    print(classification_report(y_test, y_pred_gb))

    mlflow.sklearn.log_model(model_gb, "gradient_boosting_model")